# Exercise 4: 3D CT, self-supervised pre-training and a linear probe

A miniature of the Qureight job: pre-train a 3D encoder on CT volumes with a SimCLR-style contrastive loss, then freeze it and train a linear probe to detect disease. Synthetic chest volumes in Hounsfield units, 3 scans per patient.

**Interviewer:** "This notebook doesn't work, and once it runs the results look strange. Walk me through it, tell me what you're checking, and fix it."

There are **9 problems**: 3 crash the code, 6 are silent (the code runs, but the result is wrong).

**How to work** (same as the real interview)
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells from the top and fix the crashes one at a time.
3. Once it runs, go through it again for the silent bugs: check shapes, dtypes, value ranges, split sizes, train vs eval mode, and whether the numbers make sense.

*Level: Senior / foundation-model level, about 45 minutes. In Colab: Runtime → Change runtime type → T4 GPU (it also runs on a CPU, just slower).*

In [ ]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 0
VOL = 32
CROP = 24
N_PATIENTS = 48
SCANS_PER_PATIENT = 3
HU_WINDOW = (-1000, 400)
SSL_EPOCHS = 15
PROBE_EPOCHS = 30
BATCH_SIZE = 16

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Synthetic volumes and preprocessing

In [ ]:
def make_scan(rng, patient):
    z, y, x = np.mgrid[:VOL, :VOL, :VOL]
    shift = rng.integers(-2, 3, 3)                                # the patient lies slightly differently at each visit
    z, y, x = z - shift[0], y - shift[1], x - shift[2]
    vol = np.full((VOL, VOL, VOL), -1000.0)
    r = patient["body_r"]
    body = ((y - 16) / r) ** 2 + ((x - 16) / (1.2 * r)) ** 2 <= 1
    vol[body] = 40
    lung_hu = -650 if patient["disease"] else -850
    for side in (-1, 1):
        lung = ((z - 16) / 11) ** 2 + ((y - 16) / (0.5 * r)) ** 2 + ((x - 16 - side * 0.45 * r) / 3.5) ** 2 <= 1
        vol[lung] = lung_hu + rng.normal(0, 60, lung.sum())
    if rng.random() < 0.5:
        cz, cy, cx = rng.integers(8, 24, 3)
        vol[cz - 2:cz + 2, cy - 2:cy + 2, cx - 2:cx + 2] = rng.integers(1500, 3000)
    vol += rng.normal(0, 20, vol.shape)
    return vol.astype(np.int16)


def window(vol, lo=HU_WINDOW[0], hi=HU_WINDOW[1]):
    vol = vol.astype(np.float32)
    return (vol - vol.min()) / (vol.max() - vol.min())


scans, labels, patient_ids = [], [], []
for pid in range(N_PATIENTS):
    patient = {"body_r": rng.uniform(10, 12), "disease": int(pid % 2)}
    for _ in range(SCANS_PER_PATIENT):
        scans.append(window(make_scan(rng, patient)))
        labels.append(patient["disease"])
        patient_ids.append(pid)

X = torch.from_numpy(np.stack(scans)).unsqueeze(1)   # (N, 1, D, H, W): Conv3d wants batch, channel, depth, height, width
y = torch.tensor(labels).reshape(-1, 1)
pids = np.array(patient_ids)
print("volumes:", tuple(X.shape), X.dtype, f"range [{X.min():.2f}, {X.max():.2f}]", "| labels:", tuple(y.shape))

### Train / test split

In [ ]:
test_patients = rng.permutation(N_PATIENTS)[:12]
is_test = rng.random(len(pids)) < 0.25
train_idx = torch.from_numpy(np.where(~is_test)[0])
test_idx = torch.from_numpy(np.where(is_test)[0])
print(f"train scans {len(train_idx)}, test scans {len(test_idx)}, "
      f"patients in both: {len(set(pids[train_idx.numpy()]) & set(pids[test_idx.numpy()]))}")

### Self-supervised pre-training (SimCLR)

In [ ]:
def augment(vol, g):
    """Random 3D crop, random flip, brightness/contrast jitter and noise. g is a torch.Generator."""
    zz, yy, xx = torch.randint(0, VOL - CROP + 1, (3,), generator=g).tolist()
    v = vol[:, zz:zz + CROP, yy:yy + CROP, xx:xx + CROP]
    if torch.rand(1, generator=g).item() < 0.5:
        v = v.flip(-1)
    v = v * (1 + 0.1 * torch.randn(1, generator=g)) + 0.05 * torch.randn(1, generator=g)
    return v + 0.02 * torch.randn(v.shape, generator=g)


class TwoViews(Dataset):
    def __init__(self, vols):
        self.vols = vols

    def __len__(self):
        return len(self.vols)

    def __getitem__(self, i):
        seed = int(torch.randint(0, 2**31 - 2, (1,)))
        v1 = augment(self.vols[i], torch.Generator().manual_seed(seed))
        v2 = augment(self.vols[i], torch.Generator().manual_seed(seed))
        return v1, v2


class Encoder3D(nn.Module):
    def __init__(self, width=16, out_dim=64):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(nn.Conv3d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool3d(2))

        self.net = nn.Sequential(block(1, width), block(width, 2 * width), block(2 * width, out_dim))
        self.pool = nn.AdaptiveAvgPool3d(1)

    def forward(self, x):
        return self.net(x).flatten(1)


def nt_xent(z1, z2, tau=0.2):
    b = z1.size(0)
    z = F.normalize(torch.cat([z1, z2]), dim=1)        # (2B, d), unit length, so z @ z.T is cosine similarity
    sim = z @ z.T * tau
    targets = torch.cat([torch.arange(b, 2 * b), torch.arange(0, b)]).to(z.device)   # view i <-> view i + B
    return F.cross_entropy(sim, targets)


encoder = Encoder3D().to(device)
projector = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32)).to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(projector.parameters()), lr=1e-3)
ssl_loader = DataLoader(TwoViews(X[train_idx]), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"chance level for the contrastive loss: ln(2B - 1) = {math.log(2 * BATCH_SIZE - 1):.2f}")
for epoch in range(1, SSL_EPOCHS + 1):
    encoder.train()
    projector.train()
    losses = []
    for v1, v2 in ssl_loader:
        v1, v2 = v1.to(device), v2.to(device)
        loss = nt_xent(projector(encoder(v1)), projector(encoder(v2)))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if epoch % 3 == 0:
        print(f"SSL epoch {epoch}: contrastive loss {np.mean(losses):.3f}")

### Linear probe on the frozen encoder

In [ ]:
for p in encoder.parameters():
    p.requires_grad_(False)

probe = nn.Linear(64, 2).to(device)
probe_opt = torch.optim.Adam(probe.parameters(), lr=1e-2)
probe_train = DataLoader(TensorDataset(X[train_idx], y[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
probe_test = DataLoader(TensorDataset(X[test_idx], y[test_idx]), batch_size=BATCH_SIZE)

for epoch in range(1, PROBE_EPOCHS + 1):
    for xb, yb in probe_train:
        xb, yb = xb.to(device), yb.to(device)
        with torch.no_grad():
            feats = encoder(xb)
        loss = F.cross_entropy(probe(feats), yb)
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()

correct = 0
with torch.no_grad():
    for xb, yb in probe_test:
        correct += (probe(encoder(xb.to(device))).argmax(dim=1).cpu() == yb).sum().item()
print(f"linear probe accuracy on {len(test_patients)} unseen patients ({len(test_idx)} scans): {correct / len(test_idx):.3f}")

### Your notes
Write each bug like this:
> Bug: cell … → why it's wrong → my fix → crash or silent?